In [ ]:
%pip install tensorflow
%pip install keras
%pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle

from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout # type: ignore
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
import tensorflow.keras.preprocessing.sequence # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences  # type: ignore
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [2]:
# loading data
df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

encoder = LabelEncoder()
df['label_num'] = encoder.fit_transform(df['label'])


In [3]:
#cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

df['clean_message'] = df['message'].apply(clean_text)


In [4]:
# token
vocab_size = 10000  
max_length = 150    
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"   

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(df['clean_message'])

sequences = tokenizer.texts_to_sequences(df['clean_message'])
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [ ]:
#cnn model and training
embedding_dim = 64

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    
    Conv1D(128, 5, activation='relu'),
    
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.5), 
    
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

#training
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, df['label_num'], test_size=0.2, random_state=42)

history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

model.save('cnn_spam_model.keras')


In [7]:
def predict_spam_cnn(input_text):
    cleaned_text = clean_text(input_text)
    
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding=padding_type, truncating=trunc_type)
    
    prediction_prob = model.predict(padded_sequence, verbose=0)[0][0]
    
    if prediction_prob >= 0.5:
        return "Spam"
    else:
        return "Not Spam"

test_msg = "hey my name is Tristan and you just won a free ticket to the Bahamas! Click here to claim now!"
print(f"\nTest Prediction for: '{test_msg}'")
print(f"Result: {predict_spam_cnn(test_msg)}")


Test Prediction for: 'hey my name is Tristan and you just won a free ticket to the Bahamas! Click here to claim now!'
Result: Spam
